# E15 — Online Strong-Retain Utility Router
## [해결책] E13의 실제 one-pass steering + A1의 strong-action 아이디어를 결합하되, test에서는 **cached outcome lookup 없이 실제 generation**

### 목표
E13의 핵심 장점인 **실제 runtime hidden read → action decision → L14–18 steering → 같은 KV-cache에서 답변 생성**을 유지하면서,
A1에서 확인된 아이디어인 **강한 retain 강도(0.09 / 0.12 등)를 샘플별로 선택**한다.

### 핵심 차이
- **A1**: 저장된 hidden → action 선택 → 사전 생성된 outcome 조회
- **E15**: L13 hidden을 실제 forward 중 읽음 → action/강도 선택 → L14–18에 즉시 steering → **새 답변 생성**

### 성능 목표
1. `online_utility_router`가 `noop`보다 양의 Net Gain을 낼 것.
2. `online_utility_router`가 `always retain 0.06`보다 추가 이득을 낼 것.
3. 가능하면 A1 보고값 `0.6078`을 넘길 것.  
   **주의:** A1은 C→Q/P9 계열, 본 노트북은 E11/E13의 Q→C 계열이므로 raw EM 0.6078은 참고선일 뿐 엄밀한 동등 비교값이 아니다.

### 실험 전략
- 개발 데이터에서 여러 actuator를 **실제로 generation**해 action bank를 만든다.
- hidden → action별 `ΔEM`, `P(recovery)`, `P(damage)`를 예측한다.
- 질문 단위 5-fold OOF 예측으로 utility policy의 threshold/손상 penalty/강한 retain guard를 고정한다.
- policy lock 이후 held-out test에서 **선택된 action만 한 번 실제 실행**한다.
- `noop`, `always_r0.06`, `always_r0.12`, `online router`, `random-direction matched route`를 비교한다.

## 0. [문제상황 → 원인 → 해결책 → 효과]

### 문제상황
E13 sparse gate는 실제 online generation에서 Net Gain을 냈지만, 강한 actuator를 샘플별로 고르는 기능은 제한적이었다.

### 원인
- 약한 steering만으로는 recoverable case의 ceiling이 낮을 수 있다.
- 반대로 강한 steering은 Recovery를 늘리지만 C+ Damage 위험도 증가한다.
- 따라서 `상태 분류`만이 아니라 **행별 action utility**를 직접 예측해야 한다.

### 해결책
L13 full hidden을 저차원 PCA로 압축한 뒤,
- action별 `ΔEM`
- `Recovery` 확률
- `Damage` 확률

을 함께 예측한다. 강한 retain은 C−일 가능성이 충분히 높을 때만 허용한다.

### 효과
강한 actuator의 회복력을 사용하면서 불필요한 intervention의 Damage를 억제하는 것이 목표다.

In [ ]:
# 1. 환경 설치
import sys, subprocess
TARGET_TRANSFORMERS = "4.57.1"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==" + TARGET_TRANSFORMERS,
    "accelerate", "gdown", "pyarrow", "safetensors",
    "scikit-learn==1.6.1", "joblib"
])

import os, json, time, math, re, ast as pyast, string, unicodedata, hashlib, warnings, gc
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import torch, transformers, sklearn, joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

print("torch", torch.__version__,
      "transformers", transformers.__version__,
      "sklearn", sklearn.__version__,
      "numpy", np.__version__)

In [ ]:
# 2. 공통 설정
RUN_ID = "E15_online_strong_utility_v1"

CFG = {
    "SEED": 42,
    "BATCH_SIZE": 4,
    "MAX_NEW_TOKENS": 192,
    "MAX_PROMPT_TOKENS": 4096,

    # utility representation
    "PCA_DIM": 128,
    "RIDGE_ALPHA": 20.0,
    "LOGREG_C": 0.25,

    # 실제 ||delta|| / ||hidden|| 비율
    # A1의 rho와 정의/프롬프트가 완전히 동일한 것은 아님.
    "RETAIN_RHOS": [0.05, 0.06, 0.075, 0.09, 0.10, 0.12, 0.15],
    "ACCEPT_RHOS": [0.05, 0.075],

    # 사용 가능한 vector family를 vectors.npz에서 자동 탐지
    "VECTOR_FAMILIES_PREFERRED": ["instruction", "behavior"],

    # policy OOF search
    "DAMAGE_WEIGHTS": [1.0, 1.5, 2.0, 3.0, 4.0],
    "BLENDS": [0.0, 0.5, 1.0],  # 0=gain ridge, 1=P(rec)-lambda*P(dam)
    "TAUS": [0.00, 0.01, 0.02, 0.04, 0.06, 0.08],
    "STRONG_CMINUS_GUARDS": [0.55, 0.65, 0.75, 0.85],

    # 개발셋 안전 제약
    "MAX_DAMAGE_RATE": 0.025,
    "MIN_CPLUS_DELTA": -0.01,

    "N_BOOT": 2000,
    "RUN_TEST_ORACLE_DIAGNOSTIC": False,
    "A1_RAW_REFERENCE_EM": 0.6078,
}

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
WRITE_LAYERS = tuple(range(14, 19))
CELL_NAMES = ["K+C+", "K+C-", "K-C-", "K-C+"]

np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])

print(json.dumps(CFG, indent=2))

In [ ]:
# 3. Drive 연결 + 기존 E11/E13 artifact 경로
from google.colab import drive
DRIVE = Path("/content/drive")
if not os.path.ismount(DRIVE):
    drive.mount(str(DRIVE))

P8S_ROOT = DRIVE / "MyDrive" / "p8s"
E11_ROOT = P8S_ROOT / "E11_run01"
E11_FOLD = E11_ROOT / "fold_0"
E11_QFIRST = E11_FOLD / "e11-qfirst"
E13_ROOT = P8S_ROOT / "E13_probe_attr_sparsegate_v1"
E15_ROOT = P8S_ROOT / RUN_ID
E15_ROOT.mkdir(parents=True, exist_ok=True)

required = [
    E11_FOLD / "split_manifest.csv",
    E11_QFIRST / "detectors.joblib",
    E11_QFIRST / "vectors.npz",
    E11_QFIRST / "neutral",
    E13_ROOT / "binary_probes.joblib",
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"필수 artifact가 없습니다: {p}\n"
            "먼저 E11 fold-0 및 E13을 실행했는지 확인하세요."
        )

print("E15 root:", E15_ROOT)

In [ ]:
# 4. 데이터 로드
import gdown

SOURCE_FILES = {
    "behavior": ("1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr", "p8s_behavior_v2.csv"),
    "klabels": ("1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF", "k_relabel_v2.parquet"),
}

WORK = Path("/content/e15_input")
WORK.mkdir(parents=True, exist_ok=True)

PATHS = {}
for name, (fid, fname) in SOURCE_FILES.items():
    p = WORK / fname
    if not p.exists():
        tmp = p.with_suffix(p.suffix + ".part")
        ok = gdown.download(id=fid, output=str(tmp), quiet=False)
        if not ok or not tmp.exists():
            raise RuntimeError("download failed: " + fname)
        os.replace(tmp, p)
    PATHS[name] = p

def norm_answer(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = "".join(c for c in s if c not in set(string.punctuation))
    return " ".join(x for x in s.split() if x not in {"a", "an", "the"})

def truncate_answer(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    s = s.split(". ")[0].strip().strip("\"'")
    return re.sub(r"[.,;:\s]+$", "", s).strip()

def gold_aliases(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        items = list(value)
    else:
        try:
            items = json.loads(str(value))
        except Exception:
            try:
                items = pyast.literal_eval(str(value))
            except Exception:
                items = [str(value)]
    if isinstance(items, str):
        items = [items]
    return [str(x) for x in items if norm_answer(x)]

FRAME = pd.read_csv(PATHS["behavior"]).reset_index(drop=True)
KLAB = pd.read_parquet(PATHS["klabels"])

conv = {"K+": 1, "K-": 0, "K−": 0}
yy = KLAB["kside_corr"].map(conv)
if yy.isna().any():
    raise ValueError("Unknown K label")

km = (
    pd.DataFrame({"q": KLAB.question, "k": yy.astype(int)})
    .drop_duplicates("q")
    .set_index("q").k
)
FRAME["k"] = FRAME.question.map(km).astype(int)
FRAME["c"] = FRAME.ctx_label.map({"C+": 1, "C-": 0, "C−": 0}).astype(int)
FRAME["state"] = np.where(
    FRAME.k.eq(1),
    np.where(FRAME.c.eq(1), 0, 1),
    np.where(FRAME.c.eq(1), 3, 2)
).astype(int)
FRAME["aliases"] = FRAME.gold.map(gold_aliases)
FRAME["row"] = np.arange(len(FRAME))

SPLIT = pd.read_csv(E11_FOLD / "split_manifest.csv").sort_values("row").reset_index(drop=True)
assert len(FRAME) == len(SPLIT) == 1224
assert FRAME.question.astype(str).tolist() == SPLIT.question.astype(str).tolist()

FIT = np.flatnonzero(SPLIT.role.eq("fit"))
UF = np.flatnonzero(SPLIT.role.eq("utility_fit"))
PC = np.flatnonzero(SPLIT.role.eq("policy_cal"))
TEST = np.flatnonzero(SPLIT.role.eq("test"))
DEV = np.flatnonzero(~SPLIT.role.eq("test"))

print("roles:", {r: int((SPLIT.role == r).sum()) for r in SPLIT.role.unique()})
print(pd.crosstab(FRAME.k, FRAME.c))

In [ ]:
# 5. E11 neutral hidden / probe / E13 binary probe 로드
def load_npz_shards(folder):
    shards = []
    for p in sorted(Path(folder).glob("batch_*.npz")):
        with np.load(p, allow_pickle=False) as z:
            shards.append({k: z[k].copy() for k in z.files})
    if not shards:
        raise FileNotFoundError("No NPZ shards in " + str(folder))
    return {k: np.concatenate([s[k] for s in shards], axis=0) for k in shards[0]}

E11_BANK = load_npz_shards(E11_QFIRST / "neutral")
assert np.array_equal(E11_BANK["rows"], np.arange(len(FRAME)))

H13_ALL = E11_BANK["L13"].astype(np.float32)

DET = joblib.load(E11_QFIRST / "detectors.joblib")
E11_LAST4 = DET["candidates"][(13, "last")]

BIN = joblib.load(E13_ROOT / "binary_probes.joblib")
K13 = BIN["K13"]
C13 = BIN["C13"]

def probe_aux_features(H):
    H = np.asarray(H, np.float32)
    p4 = E11_LAST4.predict_proba(H).astype(np.float32)
    pk = K13.predict_proba(H)[:, 1:2].astype(np.float32)
    pc = C13.predict_proba(H)[:, 1:2].astype(np.float32)
    ent4 = -(np.clip(p4, 1e-8, 1) * np.log(np.clip(p4, 1e-8, 1))).sum(1, keepdims=True)
    max4 = p4.max(1, keepdims=True)
    return np.concatenate([p4, pk, pc, ent4, max4], axis=1).astype(np.float32)

AUX_ALL = probe_aux_features(H13_ALL)
print("H13", H13_ALL.shape, "AUX", AUX_ALL.shape)

In [ ]:
# 6. 모델 로드 — E11과 동일 revision
from getpass import getpass
from huggingface_hub import login, model_info
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU runtime이 필요합니다.")

rev_path = E11_ROOT / "model_revision.json"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face Read token (hf_...): ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)

if rev_path.exists():
    REVISION = json.loads(rev_path.read_text())["revision"]
else:
    REVISION = model_info(MODEL_ID, token=HF_TOKEN).sha

TOK = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=HF_TOKEN, use_fast=True)
TOK.padding_side = "left"
if TOK.pad_token_id is None:
    TOK.pad_token = TOK.eos_token

MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    trust_remote_code=False,
).eval()

for p in MODEL.parameters():
    p.requires_grad_(False)

assert len(MODEL.model.layers) == 32
assert MODEL.config.hidden_size == 4096

print("GPU:", torch.cuda.get_device_name(0))
print("revision:", REVISION[:12])

In [ ]:
# 7. Q-first prompt / batching / scoring
SYSTEM_PROMPT = (
    "You are a question answering system. "
    "Answer the question based on the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
)

@dataclass(frozen=True)
class PromptInput:
    row: int
    question: str
    context: str

def safe_input(i):
    r = FRAME.iloc[int(i)]
    return PromptInput(int(i), str(r.question), str(r.ctx_text))

def render_prompt(item):
    user = "Question: " + item.question + "\nContext: " + item.context + "\nAnswer:"
    text = TOK.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    base = text.rfind(user)
    enc = TOK(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]

    qstart = base + len("Question: ")
    qend = qstart + len(item.question)
    cstart = base + len("Question: ") + len(item.question) + len("\nContext: ")

    qpos = [i for i, (s, e) in enumerate(offsets) if e > qstart and s < qend and e > s]
    if not qpos:
        raise RuntimeError("question span not found")
    if offsets[qpos[-1]][1] > cstart:
        raise AssertionError("Q-first causal span crossed Context")

    return {"ids": ids, "qidx": qpos[-1]}

def pack_rows(rows):
    device = next(MODEL.parameters()).device
    items = [safe_input(i) for i in rows]
    packs = [render_prompt(x) for x in items]
    T = max(len(x["ids"]) for x in packs)
    if T > CFG["MAX_PROMPT_TOKENS"]:
        raise ValueError(f"prompt too long: {T}")

    B = len(rows)
    ids = torch.full((B, T), TOK.pad_token_id, dtype=torch.long, device=device)
    mask = torch.zeros_like(ids)
    qidx = []
    for j, p in enumerate(packs):
        n = len(p["ids"])
        off = T - n
        ids[j, off:] = torch.tensor(p["ids"], device=device)
        mask[j, off:] = 1
        qidx.append(off + p["qidx"])
    pos = (mask.cumsum(-1) - 1).clamp_min(0)
    return ids, mask, pos, torch.tensor(qidx, device=device), items

def score_text(raw, row):
    pred = norm_answer(truncate_answer(raw))
    gs = [norm_answer(x) for x in row.aliases]
    ca = norm_answer(row.answer_surface)
    return {
        "em": int(any(pred == g for g in gs)),
        "follows_ctx": int(bool(ca) and ca in pred),
        "pred": truncate_answer(raw),
    }

In [ ]:
# 8. steering vector family 구성
with np.load(E11_QFIRST / "vectors.npz", allow_pickle=False) as z:
    keys = set(z.files)
    FAMILIES = []
    UNIT_DIRS = {}
    for fam in CFG["VECTOR_FAMILIES_PREFERRED"]:
        expected = [f"{fam}__{l}" for l in WRITE_LAYERS]
        if all(k in keys for k in expected):
            FAMILIES.append(fam)
            UNIT_DIRS[fam] = {}
            for l in WRITE_LAYERS:
                v = z[f"{fam}__{l}"].astype(np.float32)
                n = np.linalg.norm(v)
                if n < 1e-10:
                    raise ValueError(f"degenerate {fam} vector L{l}")
                UNIT_DIRS[fam][l] = (v / n).astype(np.float32)

if not FAMILIES:
    raise RuntimeError("vectors.npz에서 instruction/behavior family를 찾지 못했습니다.")

rng = np.random.default_rng(CFG["SEED"] + 909)
RANDOM_DIRS = {}
for fam in FAMILIES:
    RANDOM_DIRS[fam] = {}
    for l in WRITE_LAYERS:
        v = rng.standard_normal(MODEL.config.hidden_size).astype(np.float32)
        RANDOM_DIRS[fam][l] = v / np.linalg.norm(v)

@dataclass(frozen=True)
class Act:
    name: str
    family: str
    kind: str   # retain / accept
    rho: float

ACTIONS = []
for fam in FAMILIES:
    for rho in CFG["RETAIN_RHOS"]:
        ACTIONS.append(Act(f"{fam}_retain_r{rho:g}", fam, "retain", float(rho)))
    for rho in CFG["ACCEPT_RHOS"]:
        ACTIONS.append(Act(f"{fam}_accept_r{rho:g}", fam, "accept", float(rho)))

ACTION_BY_NAME = {a.name: a for a in ACTIONS}
print("families:", FAMILIES)
print("n actions:", len(ACTIONS))
for a in ACTIONS:
    print(a)

## 9. 실제 generation engine

중요:
- `FixedActionSession`: 개발용 action bank를 **실제로 생성**한다.
- `OnlineUtilitySession`: held-out test에서 L13 hidden을 읽고 즉시 action을 정한 뒤 L14–18에 적용한다.
- `cache`는 Colab 중단 재개용일 뿐, 다른 사람의 사전 생성 결과를 lookup하는 용도가 아니다.

In [ ]:
# 9-1. generation session / cache
class NullSession:
    def __init__(self, B):
        self.phase = "prefill"
        self.action = [None] * B
        self.max_ratio = np.zeros(B, np.float32)
        self.scores = [{} for _ in range(B)]
    def __enter__(self): return self
    def __exit__(self, *args): pass

class FixedActionSession:
    def __init__(self, B, action: Act, random_direction=False):
        self.act = action
        self.random_direction = random_direction
        self.phase = "prefill"
        self.handles = []
        self.action = [action.name] * B
        self.max_ratio = np.zeros(B, np.float32)
        self.scores = [{} for _ in range(B)]

    def __enter__(self):
        dirs = RANDOM_DIRS if self.random_direction else UNIT_DIRS
        sign = -1.0 if self.act.kind == "retain" else 1.0

        for l in WRITE_LAYERS:
            block = MODEL.model.layers[l]

            def hook(mod, args, out, ll=l):
                h = out[0] if isinstance(out, tuple) else out
                last = h[:, -1, :]
                u = torch.as_tensor(
                    dirs[self.act.family][ll],
                    device=h.device,
                    dtype=h.dtype
                )
                delta = (
                    sign * self.act.rho
                    * last.float().norm(dim=-1, keepdim=True).to(h.dtype)
                    * u[None, :]
                )
                ratio = (
                    delta.float().norm(dim=-1)
                    / last.float().norm(dim=-1).clamp_min(1e-12)
                )
                self.max_ratio = np.maximum(
                    self.max_ratio,
                    ratio.detach().cpu().numpy()
                )
                new = h.clone()
                new[:, -1, :] = last + delta
                return (new,) + out[1:] if isinstance(out, tuple) else new

            self.handles.append(block.register_forward_hook(hook))
        return self

    def __exit__(self, *args):
        for h in self.handles:
            h.remove()
        self.handles = []

@torch.inference_mode()
def generate_batch(rows, session_factory=None):
    rows = list(map(int, rows))
    ids, mask, pos, qidx, _ = pack_rows(rows)
    B = len(rows)

    eos = {TOK.eos_token_id}
    eot = TOK.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot, int) and eot >= 0 and eot != TOK.unk_token_id:
        eos.add(eot)
    eos.discard(None)
    eosids = torch.tensor(sorted(eos), device=ids.device)

    ses = session_factory(qidx) if session_factory else NullSession(B)
    pieces = []
    ended = torch.zeros(B, dtype=torch.bool, device=ids.device)
    lengths = torch.zeros(B, dtype=torch.long, device=ids.device)

    if ids.device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    with ses:
        h = MODEL.model(
            input_ids=ids,
            attention_mask=mask,
            position_ids=pos,
            use_cache=True,
            return_dict=True
        )
        past = h.past_key_values
        logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])

        for step in range(CFG["MAX_NEW_TOKENS"]):
            nxt = logits.argmax(-1)
            nxt = torch.where(ended, torch.full_like(nxt, TOK.pad_token_id), nxt)
            active = ~ended
            lengths += active.long()
            pieces.append(nxt.detach().cpu())
            ended |= torch.isin(nxt, eosids) & active

            if ended.all() or step + 1 == CFG["MAX_NEW_TOKENS"]:
                break

            ses.phase = "decode"
            mask = torch.cat(
                [mask, torch.ones((B, 1), dtype=mask.dtype, device=mask.device)],
                -1
            )
            dpos = (mask.sum(-1) - 1)[:, None]
            h = MODEL.model(
                input_ids=nxt[:, None],
                attention_mask=mask,
                position_ids=dpos,
                past_key_values=past,
                use_cache=True,
                return_dict=True
            )
            past = h.past_key_values
            logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])

    if ids.device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    texts = TOK.batch_decode(torch.stack(pieces, 1), skip_special_tokens=True)

    rec = []
    for j, (r, t) in enumerate(zip(rows, texts)):
        rec.append({
            "row": r,
            "raw": t,
            "action": (
                ses.action[j]
                if hasattr(ses, "action") and ses.action[j] is not None
                else "noop"
            ),
            "seconds": elapsed / B,
            "new_tokens": int(lengths[j]),
            "max_applied_delta_ratio": (
                float(ses.max_ratio[j]) if hasattr(ses, "max_ratio") else 0.0
            ),
            **(ses.scores[j] if hasattr(ses, "scores") else {}),
        })
    return rec

def cached_generation(tag, rows, session_factory=None):
    folder = E15_ROOT / "generation" / tag
    folder.mkdir(parents=True, exist_ok=True)
    rows = list(map(int, rows))

    for st in range(0, len(rows), CFG["BATCH_SIZE"]):
        p = folder / f"batch_{st:06d}.json"
        if p.exists():
            continue
        ix = rows[st:st + CFG["BATCH_SIZE"]]
        result = generate_batch(ix, session_factory)
        p.write_text(
            json.dumps(result, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        if st % (20 * CFG["BATCH_SIZE"]) == 0:
            print(tag, min(st + CFG["BATCH_SIZE"], len(rows)), "/", len(rows), flush=True)

    rec = [
        x
        for p in sorted(folder.glob("batch_*.json"))
        for x in json.loads(p.read_text())
    ]
    return pd.DataFrame(rec).set_index("row", drop=False).sort_index()

def attach_scores(df, rows):
    x = df.copy()
    vals = [score_text(x.loc[r, "raw"], FRAME.iloc[r]) for r in rows]
    x.loc[rows, "em"] = [v["em"] for v in vals]
    x.loc[rows, "follows_ctx"] = [v["follows_ctx"] for v in vals]
    return x

## 10. 개발 action bank — 전부 실제 generation

이 단계는 비용이 들지만 핵심이다.  
`DEV = fit + utility_fit + policy_cal` 전체에 대해 각 action을 실제 실행한다.

- test는 아직 사용하지 않는다.
- 이후 OOF utility model의 target은 `EM(action) - EM(noop)`이다.
- Colab이 끊겨도 `E15_ROOT/generation/` 아래 batch cache로 이어서 실행된다.

In [ ]:
# 10-1. DEV baseline + forced action bank
DEV_BASE = cached_generation("dev_noop", DEV)
DEV_BASE = attach_scores(DEV_BASE, DEV)

DEV_FORCED = {}
for ai, act in enumerate(ACTIONS, 1):
    print(f"[{ai}/{len(ACTIONS)}] forced {act.name}")
    df = cached_generation(
        "dev_force_" + act.name,
        DEV,
        session_factory=lambda qidx, a=act:
            FixedActionSession(len(qidx), a, random_direction=False)
    )
    DEV_FORCED[act.name] = attach_scores(df, DEV)

rows = []
b = DEV_BASE.loc[DEV, "em"].to_numpy(int)
for act in ACTIONS:
    s = DEV_FORCED[act.name].loc[DEV, "em"].to_numpy(int)
    d = s - b
    c = FRAME.iloc[DEV].c.to_numpy(int)
    rows.append({
        "action": act.name,
        "family": act.family,
        "kind": act.kind,
        "rho": act.rho,
        "EM": float(s.mean()),
        "DeltaEM": float(d.mean()),
        "Recovery": int((d == 1).sum()),
        "Damage": int((d == -1).sum()),
        "NetGain": int(d.sum()),
        "Cminus_DeltaEM": float(d[c == 0].mean()),
        "Cplus_DeltaEM": float(d[c == 1].mean()),
    })

DEV_ACTION_REPORT = pd.DataFrame(rows).sort_values(
    ["DeltaEM", "Recovery"], ascending=False
)
display(DEV_ACTION_REPORT)
DEV_ACTION_REPORT.to_csv(E15_ROOT / "DEV_ACTION_REPORT.csv", index=False)

## 11. Hidden → action utility predictor

각 action마다 세 가지를 예측한다.

1. `gain`: `EM(action)-EM(base)` 직접 회귀
2. `P(recovery)`: base 오답이 action에서 정답이 될 확률
3. `P(damage)`: base 정답이 action에서 오답이 될 확률

그리고 질문 단위 GroupKFold OOF prediction으로 policy를 선택한다.

In [ ]:
# 11-1. feature projector + utility model
class ConstantProb:
    def __init__(self, p):
        self.p = float(p)
    def predict_proba(self, X):
        p = np.full(len(X), self.p, float)
        return np.column_stack([1-p, p])

class UtilityBundle:
    def __init__(self, cfg):
        self.cfg = cfg
        self.pca = None
        self.models = {}
        self.hidden_mean_norm = None
        self.hidden_std_norm = None

    def _base_aux(self, H):
        return probe_aux_features(H)

    def fit_projector(self, H):
        H = np.asarray(H, np.float32)
        ncomp = min(self.cfg["PCA_DIM"], len(H)-1, H.shape[1])
        self.pca = PCA(
            n_components=ncomp,
            whiten=True,
            random_state=self.cfg["SEED"]
        )
        Z = self.pca.fit_transform(H).astype(np.float32)
        norms = np.linalg.norm(H, axis=1)
        self.hidden_mean_norm = float(norms.mean())
        self.hidden_std_norm = float(norms.std() + 1e-8)
        return Z

    def transform(self, H):
        H = np.asarray(H, np.float32)
        Z = self.pca.transform(H).astype(np.float32)
        aux = self._base_aux(H)
        norms = ((np.linalg.norm(H, axis=1) - self.hidden_mean_norm)
                 / self.hidden_std_norm)[:, None].astype(np.float32)
        return np.concatenate([Z, aux, norms], axis=1)

    def fit(self, H, base_em, outcome_by_action):
        self.fit_projector(H)
        X = self.transform(H)
        b = np.asarray(base_em, int)

        for act in ACTIONS:
            s = np.asarray(outcome_by_action[act.name], int)
            delta = s - b
            rec = ((b == 0) & (s == 1)).astype(int)
            dam = ((b == 1) & (s == 0)).astype(int)

            gain = make_pipeline(
                StandardScaler(),
                Ridge(alpha=self.cfg["RIDGE_ALPHA"])
            )
            gain.fit(X, delta.astype(float))

            def fit_prob(y):
                if len(np.unique(y)) < 2:
                    return ConstantProb(float(np.mean(y)))
                m = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        C=self.cfg["LOGREG_C"],
                        max_iter=1500,
                        solver="lbfgs",
                        class_weight="balanced",
                        random_state=self.cfg["SEED"]
                    )
                )
                m.fit(X, y)
                return m

            self.models[act.name] = {
                "gain": gain,
                "rec": fit_prob(rec),
                "dam": fit_prob(dam),
            }
        return self

    def predict(self, H):
        X = self.transform(H)
        n = len(X)
        A = len(ACTIONS)
        gain = np.zeros((n, A), np.float32)
        prec = np.zeros((n, A), np.float32)
        pdam = np.zeros((n, A), np.float32)

        for j, act in enumerate(ACTIONS):
            m = self.models[act.name]
            gain[:, j] = m["gain"].predict(X)
            prec[:, j] = m["rec"].predict_proba(X)[:, 1]
            pdam[:, j] = m["dam"].predict_proba(X)[:, 1]

        aux = self._base_aux(H)
        # AUX order: p4[0:4], pk=4, pc=5, ent4=6, max4=7
        pk = aux[:, 4]
        pc = aux[:, 5]
        return {
            "gain": gain,
            "prec": prec,
            "pdam": pdam,
            "pk": pk,
            "pc": pc,
        }

In [ ]:
# 11-2. DEV outcome matrix + question-wise OOF utility prediction
DEV_ROWS = np.asarray(DEV, int)
base_dev = DEV_BASE.loc[DEV_ROWS, "em"].to_numpy(int)
outcome_dev = {
    a.name: DEV_FORCED[a.name].loc[DEV_ROWS, "em"].to_numpy(int)
    for a in ACTIONS
}

groups = FRAME.iloc[DEV_ROWS].question.astype(str).to_numpy()
gkf = GroupKFold(n_splits=5)

A = len(ACTIONS)
OOF_GAIN = np.zeros((len(DEV_ROWS), A), np.float32)
OOF_PREC = np.zeros_like(OOF_GAIN)
OOF_PDAM = np.zeros_like(OOF_GAIN)
OOF_PK = np.zeros(len(DEV_ROWS), np.float32)
OOF_PC = np.zeros(len(DEV_ROWS), np.float32)

for fold, (tr, va) in enumerate(gkf.split(DEV_ROWS, groups=groups)):
    bundle = UtilityBundle(CFG).fit(
        H13_ALL[DEV_ROWS[tr]],
        base_dev[tr],
        {k: v[tr] for k, v in outcome_dev.items()}
    )
    pred = bundle.predict(H13_ALL[DEV_ROWS[va]])
    OOF_GAIN[va] = pred["gain"]
    OOF_PREC[va] = pred["prec"]
    OOF_PDAM[va] = pred["pdam"]
    OOF_PK[va] = pred["pk"]
    OOF_PC[va] = pred["pc"]
    print("OOF fold", fold, "train", len(tr), "valid", len(va))

np.savez_compressed(
    E15_ROOT / "OOF_UTILITY_PREDICTIONS.npz",
    rows=DEV_ROWS,
    gain=OOF_GAIN,
    prec=OOF_PREC,
    pdam=OOF_PDAM,
    pk=OOF_PK,
    pc=OOF_PC,
)

In [ ]:
# 12. OOF policy search
ACTION_NAMES = [a.name for a in ACTIONS]
ACTION_OUTCOME_MATRIX = np.stack([outcome_dev[n] for n in ACTION_NAMES], axis=1)

def select_actions(pred, pars):
    gain, prec, pdam = pred["gain"], pred["prec"], pred["pdam"]
    pk, pc = pred["pk"], pred["pc"]

    blend = float(pars["blend"])
    lam = float(pars["damage_weight"])
    tau = float(pars["tau"])
    cguard = float(pars["strong_cminus_guard"])

    expected = (1.0 - blend) * gain + blend * (prec - lam * pdam)
    score = expected.copy()

    for j, act in enumerate(ACTIONS):
        # strong retain: C-일 가능성이 충분히 높은 경우만 허용
        if act.kind == "retain":
            if act.rho >= 0.09:
                score[pc > (1.0 - cguard), j] = -np.inf
            elif act.rho > 0.06:
                score[pc > 0.55, j] = -np.inf

        # accept는 C+ 및 K- 쪽 guard를 강하게 둠
        if act.kind == "accept":
            score[(pc < 0.70) | (pk > 0.45), j] = -np.inf

    idx = np.argmax(score, axis=1)
    val = score[np.arange(len(score)), idx]
    chosen = np.where(np.isfinite(val) & (val > tau), idx, -1)
    return chosen, val

def evaluate_choice(chosen):
    out = base_dev.copy()
    for j in range(A):
        m = chosen == j
        out[m] = ACTION_OUTCOME_MATRIX[m, j]

    d = out - base_dev
    c = FRAME.iloc[DEV_ROWS].c.to_numpy(int)
    return {
        "EM": float(out.mean()),
        "DeltaEM": float(d.mean()),
        "Recovery": int((d == 1).sum()),
        "Damage": int((d == -1).sum()),
        "NetGain": int(d.sum()),
        "damage_rate": float(np.mean(d == -1)),
        "intervention_rate": float(np.mean(chosen >= 0)),
        "Cminus_DeltaEM": float(d[c == 0].mean()),
        "Cplus_DeltaEM": float(d[c == 1].mean()),
    }

pred_oof = {
    "gain": OOF_GAIN,
    "prec": OOF_PREC,
    "pdam": OOF_PDAM,
    "pk": OOF_PK,
    "pc": OOF_PC,
}

search = []
for blend in CFG["BLENDS"]:
    for dw in CFG["DAMAGE_WEIGHTS"]:
        for tau in CFG["TAUS"]:
            for guard in CFG["STRONG_CMINUS_GUARDS"]:
                pars = {
                    "blend": blend,
                    "damage_weight": dw,
                    "tau": tau,
                    "strong_cminus_guard": guard,
                }
                chosen, _ = select_actions(pred_oof, pars)
                met = evaluate_choice(chosen)
                met["feasible"] = (
                    met["damage_rate"] <= CFG["MAX_DAMAGE_RATE"]
                    and met["Cplus_DeltaEM"] >= CFG["MIN_CPLUS_DELTA"]
                    and met["NetGain"] > 0
                )
                search.append({**pars, **met})

POLICY_SEARCH = pd.DataFrame(search)
feas = POLICY_SEARCH[POLICY_SEARCH.feasible].copy()

if len(feas) == 0:
    raise RuntimeError(
        "OOF에서 positive feasible policy가 없습니다. "
        "강도 grid/feature/action family를 먼저 수정하세요."
    )

feas = feas.sort_values(
    ["EM", "NetGain", "Damage", "intervention_rate"],
    ascending=[False, False, True, True]
)
best = feas.iloc[0]

POLICY = {
    "blend": float(best["blend"]),
    "damage_weight": float(best["damage_weight"]),
    "tau": float(best["tau"]),
    "strong_cminus_guard": float(best["strong_cminus_guard"]),
}

display(feas.head(20))
print("LOCKED POLICY:", json.dumps(POLICY, indent=2))

POLICY_SEARCH.to_csv(E15_ROOT / "OOF_POLICY_SEARCH.csv", index=False)

In [ ]:
# 13. 최종 utility predictor를 전체 DEV로 재학습하고 policy lock 저장
FINAL_BUNDLE = UtilityBundle(CFG).fit(
    H13_ALL[DEV_ROWS],
    base_dev,
    outcome_dev,
)

MODEL_PATH = E15_ROOT / "FINAL_UTILITY_BUNDLE.joblib"
joblib.dump(FINAL_BUNDLE, MODEL_PATH)

POLICY_LOCK = {
    "created_before_test_generation": True,
    "test_used_for_selection": False,
    "policy": POLICY,
    "actions": [asdict(a) for a in ACTIONS],
    "vector_families": FAMILIES,
    "rho_definition": "actual ||delta|| / ||hidden|| at each write layer",
    "read_layer": 13,
    "write_layers": list(WRITE_LAYERS),
    "utility_features": "PCA(L13 hidden)+4state probs+K/C probs+entropy+confidence+hidden norm",
    "OOF_grouping": "question-level 5-fold on all non-test development rows",
}
LOCK_PATH = E15_ROOT / "POLICY_LOCK.json"
LOCK_PATH.write_text(
    json.dumps(POLICY_LOCK, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
LOCK_HASH = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()

print("POLICY LOCK HASH:", LOCK_HASH[:16])

## 14. Held-out test — 실제 online action/강도 선택

`OnlineUtilitySession`은 L13 output을 받은 순간 현재 batch의 hidden으로 utility를 계산한다.
그 후 L14–18에서 선택된 family/direction/rho를 실제 적용한다.

**중요:** test의 사전 생성 action outcome을 조회하지 않는다.

In [ ]:
# 14-1. Online runtime router
class OnlineUtilitySession:
    def __init__(self, qidx, bundle, policy, random_direction=False):
        self.qidx = qidx
        self.bundle = bundle
        self.policy = policy
        self.random_direction = random_direction
        self.phase = "prefill"
        self.handles = []
        B = len(qidx)
        self.action = [None] * B
        self.max_ratio = np.zeros(B, np.float32)
        self.scores = [{} for _ in range(B)]

    def __enter__(self):
        block13 = MODEL.model.layers[13]

        def read_hook(mod, args, out):
            if self.phase != "prefill":
                return out
            h = out[0] if isinstance(out, tuple) else out
            last = h[:, -1, :].detach().float().cpu().numpy()

            pred = self.bundle.predict(last)
            chosen, val = select_actions(pred, self.policy)

            for j in range(len(chosen)):
                self.scores[j] = {
                    "policy_score": float(val[j]),
                    "pK13": float(pred["pk"][j]),
                    "pC13": float(pred["pc"][j]),
                }
                if chosen[j] >= 0:
                    self.action[j] = ACTIONS[int(chosen[j])].name
            return out

        self.handles.append(block13.register_forward_hook(read_hook))

        dirs = RANDOM_DIRS if self.random_direction else UNIT_DIRS

        for l in WRITE_LAYERS:
            block = MODEL.model.layers[l]

            def write_hook(mod, args, out, ll=l):
                h = out[0] if isinstance(out, tuple) else out
                last = h[:, -1, :]
                delta = torch.zeros_like(last)

                for j, name in enumerate(self.action):
                    if name is None:
                        continue
                    act = ACTION_BY_NAME[name]
                    sign = -1.0 if act.kind == "retain" else 1.0
                    u = torch.as_tensor(
                        dirs[act.family][ll],
                        device=h.device,
                        dtype=h.dtype
                    )
                    delta[j] = (
                        sign * act.rho
                        * last[j].float().norm().to(h.dtype)
                        * u
                    )

                if not torch.any(delta):
                    return out

                ratio = (
                    delta.float().norm(dim=-1)
                    / last.float().norm(dim=-1).clamp_min(1e-12)
                )
                self.max_ratio = np.maximum(
                    self.max_ratio,
                    ratio.detach().cpu().numpy()
                )

                new = h.clone()
                new[:, -1, :] = last + delta
                return (new,) + out[1:] if isinstance(out, tuple) else new

            self.handles.append(block.register_forward_hook(write_hook))

        return self

    def __exit__(self, *args):
        for h in self.handles:
            h.remove()
        self.handles = []

In [ ]:
# 15. Test generation: noop / always .06 / always .12 / online / random matched
assert hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest() == LOCK_HASH

TEST_BASE = cached_generation("test_noop", TEST)
TEST_BASE = attach_scores(TEST_BASE, TEST)

BASE_FAMILY = "instruction" if "instruction" in FAMILIES else FAMILIES[0]

def find_action(family, kind, rho):
    for a in ACTIONS:
        if a.family == family and a.kind == kind and abs(a.rho - rho) < 1e-9:
            return a
    return None

A06 = find_action(BASE_FAMILY, "retain", 0.06)
A12 = find_action(BASE_FAMILY, "retain", 0.12)

if A06 is None or A12 is None:
    raise RuntimeError("always .06/.12 비교 action을 찾지 못했습니다.")

TEST_ALWAYS06 = cached_generation(
    "test_always_r006",
    TEST,
    session_factory=lambda qidx:
        FixedActionSession(len(qidx), A06, random_direction=False)
)
TEST_ALWAYS06 = attach_scores(TEST_ALWAYS06, TEST)

TEST_ALWAYS12 = cached_generation(
    "test_always_r012",
    TEST,
    session_factory=lambda qidx:
        FixedActionSession(len(qidx), A12, random_direction=False)
)
TEST_ALWAYS12 = attach_scores(TEST_ALWAYS12, TEST)

TEST_ONLINE = cached_generation(
    "test_online_utility_router",
    TEST,
    session_factory=lambda qidx:
        OnlineUtilitySession(qidx, FINAL_BUNDLE, POLICY, random_direction=False)
)
TEST_ONLINE = attach_scores(TEST_ONLINE, TEST)

TEST_RANDOM = cached_generation(
    "test_online_router_random_direction",
    TEST,
    session_factory=lambda qidx:
        OnlineUtilitySession(qidx, FINAL_BUNDLE, POLICY, random_direction=True)
)
TEST_RANDOM = attach_scores(TEST_RANDOM, TEST)

assert np.array_equal(
    TEST_ONLINE.loc[TEST, "action"].to_numpy(),
    TEST_RANDOM.loc[TEST, "action"].to_numpy()
), "random-direction control의 route가 달라졌습니다."

assert hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest() == LOCK_HASH
print("test generation complete")

In [ ]:
# 16. 최종 평가 + bootstrap CI
def question_bootstrap_delta(ref, active, rows, n_boot=2000, seed=42):
    rows = np.asarray(rows, int)
    r = ref.loc[rows, "em"].to_numpy(int)
    a = active.loc[rows, "em"].to_numpy(int)
    d = a - r
    q = FRAME.iloc[rows].question.astype(str).to_numpy()
    uq = np.unique(q)
    lookup = {x: np.flatnonzero(q == x) for x in uq}
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        ix = np.concatenate([lookup[x] for x in rng.choice(uq, len(uq), replace=True)])
        vals.append(float(d[ix].mean()))
    lo, hi = np.quantile(vals, [0.025, 0.975])
    return float(d.mean()), float(lo), float(hi)

def qa_metrics(base, active, name):
    rows = np.asarray(TEST, int)
    b = base.loc[rows, "em"].to_numpy(int)
    s = active.loc[rows, "em"].to_numpy(int)
    d = s - b

    out = {
        "method": name,
        "n": len(rows),
        "baseline_EM": float(b.mean()),
        "steered_EM": float(s.mean()),
        "Recovery": int((d == 1).sum()),
        "Damage": int((d == -1).sum()),
        "NetGain": int(d.sum()),
        "DeltaEM": float(d.mean()),
        "intervention_rate": float(active.loc[rows, "action"].ne("noop").mean()),
        "max_applied_delta_ratio": float(active.loc[rows, "max_applied_delta_ratio"].max()),
        "latency_seconds_mean": float(active.loc[rows, "seconds"].mean()),
    }

    state = FRAME.iloc[rows].state.to_numpy()
    for st, nm in enumerate(CELL_NAMES):
        m = state == st
        out[nm + "_DeltaEM"] = float(d[m].mean())
        out[nm + "_n"] = int(m.sum())

    return out

FINAL = pd.DataFrame([
    qa_metrics(TEST_BASE, TEST_BASE, "noop"),
    qa_metrics(TEST_BASE, TEST_ALWAYS06, f"always_{BASE_FAMILY}_retain_r0.06"),
    qa_metrics(TEST_BASE, TEST_ALWAYS12, f"always_{BASE_FAMILY}_retain_r0.12"),
    qa_metrics(TEST_BASE, TEST_ONLINE, "E15_online_utility_router"),
    qa_metrics(TEST_BASE, TEST_RANDOM, "E15_random_direction_same_route"),
])

delta06, lo06, hi06 = question_bootstrap_delta(
    TEST_ALWAYS06, TEST_ONLINE, TEST, CFG["N_BOOT"], CFG["SEED"]
)
delta_base, lob, hib = question_bootstrap_delta(
    TEST_BASE, TEST_ONLINE, TEST, CFG["N_BOOT"], CFG["SEED"] + 1
)

FINAL["vs_always06_DeltaEM"] = np.nan
FINAL["vs_always06_CI_low"] = np.nan
FINAL["vs_always06_CI_high"] = np.nan

m = FINAL.method.eq("E15_online_utility_router")
FINAL.loc[m, "vs_always06_DeltaEM"] = delta06
FINAL.loc[m, "vs_always06_CI_low"] = lo06
FINAL.loc[m, "vs_always06_CI_high"] = hi06

display(FINAL)
FINAL.to_csv(E15_ROOT / "FINAL_QA_SUMMARY.csv", index=False)

online_em = float(FINAL.loc[m, "steered_EM"].iloc[0])
always06_em = float(
    FINAL.loc[FINAL.method.str.contains("always_.*r0.06", regex=True), "steered_EM"].iloc[0]
)

print("\n===== 핵심 판정 =====")
print(f"E15 online EM      : {online_em:.4f}")
print(f"Always .06 EM      : {always06_em:.4f}")
print(f"vs Always .06      : {delta06*100:+.2f}%p [{lo06*100:+.2f}, {hi06*100:+.2f}]")
print(f"vs noop            : {delta_base*100:+.2f}%p [{lob*100:+.2f}, {hib*100:+.2f}]")
print(f"A1 raw ref 0.6078  : {'BEAT' if online_em > CFG['A1_RAW_REFERENCE_EM'] else 'NOT YET'}")
print("※ A1 raw ref는 protocol이 달라 참고선일 뿐 직접 통계 비교값이 아닙니다.")